# Optimización del aprendizaje de los modelos seleccionados
## Spotify Tracks: búsqueda de hiperparámetros, esquemas de validación cruzada y ensambles

**Asignatura:** Machine Learning (MLY1101) · **Institución:** Duoc UC
**Contexto:** preparación para la Evaluación N.º 3 (continuación de la EP2)
**Integrantes:** Alejandra González, Constanza González, Diego Villar.

En la EP2 se eligieron dos modelos ganadores, ambos **HistGradientBoosting**:

- **Regresión** (popularidad 0–100): MAE 9,24 en prueba.
- **Clasificación** (bajo / medio / alto, sin balanceo): ROC-AUC macro 0,910 en prueba.

Este notebook prueba distintos **métodos de optimización del aprendizaje** sobre esos dos
modelos y elige, para cada tarea, la combinación que más los mejora:

| Dimensión | Opciones probadas |
|---|---|
| Método de ajuste | `cross_validate` (validación cruzada de una configuración fija), `GridSearchCV`, `RandomizedSearchCV` |
| Esquema de validación cruzada | `KFold` estándar, `StratifiedKFold` estratificado, `GroupKFold` agrupado (referencia de la EP2), `StratifiedGroupKFold`, `LeaveOneOut` |
| Estrategia de ensamble | **Boosting** (HistGradientBoosting, el modelo actual) y **Bagging** (`Bagging*` sobre ese mismo modelo) |

> **Tiempo de ejecución:** la primera ejecución completa tarda alrededor de 2 horas en un equipo de
> 4 núcleos. Cada experimento guarda su resultado en `data/processed/ep3_cache/`; con
> `USAR_CACHE = True` (por defecto) una nueva ejecución reutiliza esos resultados y termina en
> pocos minutos. Para recalcular todo, borre esa carpeta o use `USAR_CACHE = False`.

## 1. Diseño experimental

### 1.1 Qué hace cada método

| Método | Qué hace | Costo |
|---|---|---|
| **CrossValidation** (`cross_validate`) | Evalúa **una configuración fija** en *k* pliegues: entrena con *k−1* y mide en el restante. No busca hiperparámetros; estima el desempeño. | *k* ajustes |
| **GridSearchCV** | Prueba **todas** las combinaciones de una grilla de valores, cada una con validación cruzada, y se queda con la mejor. | (n.º de combinaciones) × *k* ajustes |
| **RandomizedSearchCV** | Prueba **n configuraciones al azar** desde distribuciones. Con el mismo presupuesto que la grilla puede explorar más hiperparámetros y valores intermedios. | n × *k* ajustes |

| Esquema de validación | Cómo arma los pliegues | Riesgo en este dataset |
|---|---|---|
| **KFold estándar** | Reparte las canciones al azar. | Ignora las reediciones: una misma canción (artista + título) puede quedar en entrenamiento y validación → estimación **optimista**. |
| **StratifiedKFold** | Como KFold, pero mantiene la proporción de cada estrato en todos los pliegues: clases bajo/medio/alto en clasificación, **deciles de popularidad** en regresión. | Igual que KFold: no respeta las reediciones. |
| **GroupKFold** | Una canción y sus reediciones (misma `clave_cancion`) quedan siempre en el mismo pliegue. | Sin fuga. Es el esquema de la EP2. |
| **StratifiedGroupKFold** | Agrupa y además intenta estratificar. | Sin fuga. |
| **LeaveOneOut (LOO)** | *n* pliegues de una sola canción. | Con 57.392 canciones serían 57.392 ajustes por configuración (≈ 6 días por cada una): **inviable**. Se aplica a una submuestra de 500 canciones para comparar su estimación con KFold. |

| Estrategia | Idea | Cómo se aplica aquí |
|---|---|---|
| **Boosting** | Árboles **en secuencia**; cada uno corrige el error de los anteriores. Reduce sobre todo el **sesgo**. | Es el modelo seleccionado (HistGradientBoosting). Se ajustan sus hiperparámetros. |
| **Bagging** | Entrena varias copias del modelo con **muestras bootstrap** distintas y promedia. Reduce sobre todo la **varianza**. | `BaggingRegressor` / `BaggingClassifier` con el HistGradientBoosting de la EP2 como modelo base. |

### 1.2 Cómo se comparan las combinaciones sin hacer trampa

Las estimaciones de validación cruzada **no son comparables entre esquemas**: KFold y
StratifiedKFold pueden verse mejores solo porque tienen fuga de reediciones. Por eso se usa
una vara común, separada antes de todo:

1. El **entrenamiento de la EP2** (71.740 canciones) se divide, agrupando por artista + título, en
   - **ajuste** (80%): aquí corren todas las búsquedas y validaciones cruzadas;
   - **validación** (20%): ninguna búsqueda la ve. Cada combinación se reentrena con todo el
     conjunto de ajuste y se mide **una vez** aquí.
2. Se elige la combinación ganadora por su resultado en **validación**, con la métrica principal
   de la EP2: **MAE** en regresión y **ROC-AUC macro** en clasificación (redondeados a 2 y 3
   decimales; los empates se resuelven con RMSE o F1 macro, y luego con el menor tiempo).
3. La ganadora se reentrena con las 71.740 canciones y se evalúa **una sola vez** en el
   conjunto de **prueba** de la EP2 (17.843 canciones), junto al modelo de la EP2.

### 1.3 Combinaciones evaluadas

| Bloque | Combinaciones | Para qué |
|---|---|---|
| A. CrossValidation | Modelo EP2 × {KFold, StratifiedKFold, GroupKFold, StratifiedGroupKFold} | Medir cuánto se equivoca cada esquema al **estimar** el desempeño. |
| B. LeaveOneOut | Modelo EP2 en 500 canciones × {LOO, KFold 5, KFold 10, GroupKFold 5} | Comparar LOO con KFold donde LOO es factible. |
| C. Búsqueda + esquema (Boosting) | {GridSearchCV, RandomizedSearchCV} × {KFold, StratifiedKFold, GroupKFold} | Ver si el método de búsqueda y el esquema cambian los hiperparámetros elegidos y su desempeño real. |
| D. Bagging vs. Boosting | {GridSearchCV, RandomizedSearchCV} × GroupKFold, con Bagging | Ver si promediar varios modelos mejora al modelo de boosting. |

**Combinaciones excluidas, y por qué:**

- **Búsqueda con LOO:** multiplicaría por 57.392 el costo de cada configuración.
- **Bagging con KFold y StratifiedKFold:** el efecto del esquema se mide en el bloque C. Repetirlo con
  Bagging, que es entre 5 y 10 veces más costoso, no aporta información nueva, y esos esquemas no son
  adecuados para elegir hiperparámetros en este dataset (tienen fuga, ver bloque A).

Todas las validaciones cruzadas usan **3 pliegues**, como en la EP2, salvo el bloque B.

## 2. Preparación

Se reutilizan la base de modelamiento y la partición de la EP2 (`spotify_multigenero.csv` y
`particion_modelo.csv`), el mismo preprocesamiento (incluido el *target encoding* del artista,
dentro del pipeline) y los hiperparámetros de los modelos guardados en `models/`.

In [ ]:
from pathlib import Path
import re
import sys
import tempfile
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (BaggingClassifier, BaggingRegressor,
                              HistGradientBoostingClassifier, HistGradientBoostingRegressor)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, f1_score, make_scorer, mean_absolute_error, precision_score,
                             r2_score, recall_score, roc_auc_score, root_mean_squared_error)
from sklearn.model_selection import (GridSearchCV, GroupKFold, GroupShuffleSplit, KFold, LeaveOneOut,
                                     RandomizedSearchCV, StratifiedGroupKFold, StratifiedKFold,
                                     cross_val_predict, cross_validate)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
SEMILLA = 42
N_PLIEGUES = 3
USAR_CACHE = True   # False = recalcular todos los experimentos

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

carpeta_actual = Path.cwd()
PROJECT_ROOT = carpeta_actual.parent if carpeta_actual.name == 'notebooks' else carpeta_actual
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
IMAGES_DIR = PROJECT_ROOT / 'images'
MODELS_DIR = PROJECT_ROOT / 'models'
CACHE_DIR = PROCESSED_DIR / 'ep3_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))
from src.transformadores import CodificadorArtista  # noqa: E402

print('Proyecto:', PROJECT_ROOT)

In [ ]:
# Base de modelamiento EP2 (1 canción = 1 fila, géneros multi-hot) y su partición
df_modelo = pd.read_csv(PROCESSED_DIR / 'spotify_multigenero.csv')
particion = pd.read_csv(PROCESSED_DIR / 'particion_modelo.csv')


def normalizar_texto(serie):
    return serie.astype('string').str.lower().str.strip().str.replace(r'\s+', ' ', regex=True)


df_modelo['clave_cancion'] = normalizar_texto(df_modelo['artistas']) + ' || ' + normalizar_texto(df_modelo['nombre_cancion'])
assert (df_modelo['id_cancion'].to_numpy() == particion['id_cancion'].to_numpy()).all()
assert (df_modelo['clave_cancion'].fillna('').astype(str).to_numpy()
        == particion['clave_cancion'].fillna('').astype(str).to_numpy()).all()
assert (df_modelo['particion'].to_numpy() == particion['particion'].to_numpy()).all()

variables_numericas = ['duracion_ms', 'bailabilidad', 'energia', 'volumen_db', 'presencia_habla', 'acusticidad',
                       'instrumentalidad', 'presencia_en_vivo', 'positividad', 'tempo_bpm']
variables_categoricas = ['contenido_explicito', 'tonalidad', 'modo', 'compas']
columnas_genero = [c for c in df_modelo.columns if c.startswith('genero_') and c != 'genero_principal']
variables_artista = ['artistas', 'nombre_cancion']
variables_modelo = variables_numericas + variables_categoricas + columnas_genero + variables_artista
MAPA_CLASES = {'bajo': 0, 'medio': 1, 'alto': 2}

# Modelos ganadores de la EP2 (pipelines completos ya entrenados)
modelo_ep2 = {'regresion': joblib.load(MODELS_DIR / 'regresor_popularidad.joblib'),
              'clasificacion': joblib.load(MODELS_DIR / 'clasificador_popularidad.joblib')}
for pipe in modelo_ep2.values():
    assert list(pipe.named_steps['prep'].feature_names_in_) == variables_modelo


def crear_preprocesador():
    """Mismo ColumnTransformer de la EP2 (sin ajustar)."""
    categoricas = Pipeline([('imputador', SimpleImputer(strategy='most_frequent')),
                            ('codificador', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    return ColumnTransformer([
        ('numericas', StandardScaler(), variables_numericas),
        ('categoricas', categoricas, variables_categoricas),
        ('generos', 'passthrough', columnas_genero),
        ('artista', Pipeline([('target_encoding', CodificadorArtista(suavizado=10, n_pliegues=5, mapa_clases=MAPA_CLASES)),
                              ('escalador', StandardScaler())]), variables_artista),
    ])


es_train = df_modelo['particion'].eq('entrenamiento').to_numpy()
X = df_modelo[variables_modelo]
y_reg = df_modelo['popularidad']
y_clf = pd.cut(y_reg, bins=[-np.inf, 30, 60, np.inf], right=False, labels=['bajo', 'medio', 'alto']).astype(str)
grupos = df_modelo['clave_cancion'].fillna('').astype(str)

X_train, X_test = X[es_train], X[~es_train]
grupos_train = grupos[es_train]

# Vara común: 80% ajuste / 20% validación dentro del entrenamiento, agrupado por artista + título
divisor = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEMILLA)
i_aj, i_val = next(divisor.split(X_train, groups=grupos_train))
X_aj, X_val = X_train.iloc[i_aj], X_train.iloc[i_val]
grupos_aj = grupos_train.iloc[i_aj]
assert not set(grupos_aj) & set(grupos_train.iloc[i_val])

print(f'Entrenamiento EP2: {len(X_train):,} | prueba EP2: {len(X_test):,}')
print(f'  -> ajuste (búsquedas y CV): {len(X_aj):,} | validación (vara común): {len(X_val):,} | '
      f'claves artista+título compartidas: 0')

In [ ]:
recall_alto = make_scorer(recall_score, labels=['alto'], average='macro', zero_division=0)


def tarea_config(nombre, y, scoring, principal, mayor_es_mejor, secundaria, bagging, estratos_aj):
    y_tr = y[es_train]
    return {'nombre': nombre, 'y_aj': y_tr.iloc[i_aj], 'y_val': y_tr.iloc[i_val], 'y_train': y_tr,
            'y_test': y[~es_train], 'scoring': scoring, 'principal': principal, 'mayor_es_mejor': mayor_es_mejor,
            'secundaria': secundaria, 'bagging': bagging, 'estratos_aj': estratos_aj}


TAREAS = {
    'regresion': tarea_config(
        'Regresión', y_reg,
        {'MAE': 'neg_mean_absolute_error', 'RMSE': 'neg_root_mean_squared_error', 'R2': 'r2'},
        'MAE', False, 'RMSE', BaggingRegressor,
        # estratos para StratifiedKFold en regresión: deciles de popularidad
        pd.qcut(y_reg[es_train].iloc[i_aj], 10, labels=False, duplicates='drop').to_numpy()),
    'clasificacion': tarea_config(
        'Clasificación', y_clf,
        {'AUC': 'roc_auc_ovr', 'F1_macro': 'f1_macro', 'Recall_alto': recall_alto},
        'AUC', True, 'F1_macro', BaggingClassifier,
        y_clf[es_train].iloc[i_aj].to_numpy()),
}
for tarea, cfg in TAREAS.items():
    cfg['modelo_base'] = clone(modelo_ep2[tarea].named_steps['modelo'])
    params = {k: v for k, v in cfg['modelo_base'].get_params().items()
              if k in ('learning_rate', 'max_iter', 'max_leaf_nodes', 'min_samples_leaf', 'l2_regularization')}
    cfg['params_ep2'] = params
    print(f"{cfg['nombre']}: {type(cfg['modelo_base']).__name__} con hiperparámetros EP2 {params}")

### 2.1 Funciones comunes

- `crear_esquemas`: arma los pliegues de cada esquema sobre el conjunto de ajuste y mide la **fuga**:
  el porcentaje de canciones de validación cuya clave artista + título también está en el
  entrenamiento del pliegue.
- `metricas_validacion`: mide un modelo ya entrenado en el conjunto de validación (la vara común).
- `ejecutar_busqueda`: corre `GridSearchCV` o `RandomizedSearchCV`. El pipeline usa `memory` para
  no recalcular el preprocesamiento de un mismo pliegue en cada configuración.
- `con_cache`: guarda el resultado de cada experimento en `data/processed/ep3_cache/`.

In [ ]:
def slug(texto):
    return re.sub(r'[^a-z0-9]+', '_', texto.lower()).strip('_')


def con_cache(clave, funcion):
    ruta = CACHE_DIR / f'{clave}.joblib'
    if USAR_CACHE and ruta.exists():
        resultado = joblib.load(ruta)
        print(f'[caché] {clave} (cálculo original: {resultado["segundos"]:.0f} s)')
        return resultado
    inicio = time.time()
    resultado = funcion()
    resultado['segundos'] = round(time.time() - inicio, 1)
    joblib.dump(resultado, ruta)
    print(f'[calculado] {clave}: {resultado["segundos"]:.0f} s')
    return resultado


def crear_pipeline(modelo, memoria=None):
    return Pipeline([('prep', crear_preprocesador()), ('modelo', modelo)], memory=memoria)


def crear_esquemas(tarea, n=N_PLIEGUES):
    estratos = TAREAS[tarea]['estratos_aj']
    return {
        'KFold (estándar)': list(KFold(n, shuffle=True, random_state=SEMILLA).split(X_aj)),
        'StratifiedKFold (estratificado)': list(
            StratifiedKFold(n, shuffle=True, random_state=SEMILLA).split(X_aj, estratos)),
        'GroupKFold (agrupado)': list(GroupKFold(n).split(X_aj, groups=grupos_aj)),
        'StratifiedGroupKFold (estratificado y agrupado)': list(
            StratifiedGroupKFold(n, shuffle=True, random_state=SEMILLA).split(X_aj, estratos, grupos_aj)),
    }


def porcentaje_fuga(splits, g=grupos_aj):
    g = np.asarray(g)
    return 100 * np.mean([np.isin(g[va], g[tr]).mean() for tr, va in splits])


def signo(scoring, metrica):
    criterio = scoring[metrica]
    return -1 if isinstance(criterio, str) and criterio.startswith('neg_') else 1


def metricas_desde_prediccion(tarea, y_real, pred, clases=None):
    if tarea == 'regresion':
        return {'MAE': mean_absolute_error(y_real, pred), 'RMSE': root_mean_squared_error(y_real, pred),
                'R2': r2_score(y_real, pred)}
    proba = pred
    clase_pred = np.asarray(clases)[proba.argmax(axis=1)]
    return {'AUC': roc_auc_score(y_real, proba, multi_class='ovr', labels=clases),
            'F1_macro': f1_score(y_real, clase_pred, average='macro'),
            'Accuracy': accuracy_score(y_real, clase_pred),
            'Recall_alto': recall_score(y_real, clase_pred, labels=['alto'], average='macro', zero_division=0),
            'Precision_alto': precision_score(y_real, clase_pred, labels=['alto'], average='macro', zero_division=0)}


def metricas_validacion(tarea, modelo, X_, y_):
    if tarea == 'regresion':
        return metricas_desde_prediccion(tarea, y_, modelo.predict(X_))
    return metricas_desde_prediccion(tarea, y_, modelo.predict_proba(X_), modelo.classes_)


def formatear_params(params):
    partes = []
    for k, v in params.items():
        k = k.replace('modelo__', '')
        if isinstance(v, (float, np.floating)):
            v = f'{v:.3g}'
        partes.append(f'{k}={v}')
    return ', '.join(partes)


print('Funciones listas.')

## 3. Bloque A — CrossValidation: ¿qué esquema estima mejor el desempeño real?

Se evalúa el **modelo de la EP2, con sus hiperparámetros fijos**, con `cross_validate` en los
cuatro esquemas de 3 pliegues sobre el conjunto de ajuste. Luego se compara cada estimación con
el **desempeño real** del mismo modelo entrenado con todo el ajuste y medido en validación.

- **Optimismo** = cuánto mejor se ve el modelo en la validación cruzada que en validación
  (MAE: validación − CV; AUC: CV − validación). Positivo = la validación cruzada promete más de
  lo que el modelo entrega.

In [ ]:
filas_A = []
referencia_val = {}
for tarea, cfg in TAREAS.items():
    for nombre_esquema, splits in crear_esquemas(tarea).items():
        def calcular(splits=splits):
            r = cross_validate(crear_pipeline(clone(cfg['modelo_base'])), X_aj, cfg['y_aj'], cv=splits,
                               scoring=cfg['scoring'], error_score='raise')
            return {**{f'{m}_cv': signo(cfg['scoring'], m) * r[f'test_{m}'].mean() for m in cfg['scoring']},
                    **{f'{m}_std': r[f'test_{m}'].std() for m in cfg['scoring']}}
        r = con_cache(f'A_{tarea}_{slug(nombre_esquema)}', calcular)
        filas_A.append({'tarea': cfg['nombre'], 'esquema': nombre_esquema, 'fuga_%': porcentaje_fuga(splits),
                        **{k: v for k, v in r.items()}})
    referencia_val[tarea] = con_cache(
        f'A_{tarea}_referencia_validacion',
        lambda: metricas_validacion(tarea, crear_pipeline(clone(cfg['modelo_base'])).fit(X_aj, cfg['y_aj']),
                                    X_val, cfg['y_val']))

resultados_A = pd.DataFrame(filas_A)
for tarea, cfg in TAREAS.items():
    p = cfg['principal']
    sub = resultados_A[resultados_A['tarea'] == cfg['nombre']].copy()
    real = referencia_val[tarea][p]
    sub[f'{p}_validacion_real'] = real
    sub['optimismo'] = (sub[f'{p}_cv'] - real) if cfg['mayor_es_mejor'] else (real - sub[f'{p}_cv'])
    resultados_A.loc[sub.index, f'{p}_validacion_real'] = real
    resultados_A.loc[sub.index, 'optimismo'] = sub['optimismo']
    print(f"\n{cfg['nombre']} — modelo EP2 con hiperparámetros fijos ({p}; validación real = {real:.4f})")
    display(sub[['esquema', 'fuga_%', f'{p}_cv', f'{p}_std', f'{p}_validacion_real', 'optimismo', 'segundos']]
            .set_index('esquema').round(4))
resultados_A.to_csv(PROCESSED_DIR / 'ep3_bloque_A_esquemas_cv.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
for ax, (tarea, cfg) in zip(axes, TAREAS.items()):
    p = cfg['principal']
    sub = resultados_A[resultados_A['tarea'] == cfg['nombre']]
    etiquetas = [e.split(' (')[0] for e in sub['esquema']]
    colores = ['tab:red' if f > 1 else 'tab:green' for f in sub['fuga_%']]
    ax.bar(etiquetas, sub[f'{p}_cv'], yerr=sub[f'{p}_std'], color=colores, alpha=0.8, capsize=4)
    real = referencia_val[tarea][p]
    ax.axhline(real, color='black', linestyle='--', label=f'Validación real = {real:.3f}')
    for x, (v, f) in enumerate(zip(sub[f'{p}_cv'], sub['fuga_%'])):
        ax.text(x, v, f'{v:.3f}\nfuga {f:.1f}%', ha='center', va='bottom', fontsize=8)
    margen = (sub[f'{p}_cv'].max() - sub[f'{p}_cv'].min()) + abs(real) * 0.01
    ax.set_ylim(min(sub[f'{p}_cv'].min(), real) - 2 * margen, max(sub[f'{p}_cv'].max(), real) + 2 * margen)
    ax.set_title(f"{cfg['nombre']}: {p} estimado por CV (rojo = con fuga)", fontweight='bold')
    ax.set_ylabel(p); ax.legend(loc='lower right'); ax.tick_params(axis='x', labelsize=8)
plt.tight_layout()
plt.savefig(IMAGES_DIR / '30_ep3_esquemas_cv.png', dpi=150, bbox_inches='tight')
plt.show()

<!-- INTERP_A -->

## 4. Bloque B — LeaveOneOut en una submuestra

LOO entrena el modelo *n* veces, dejando fuera una canción cada vez. Sobre las 57.392 canciones de
ajuste es inviable, así que se toma una **submuestra aleatoria de 500 canciones** y se compara la
estimación de LOO con la de KFold de 5 y 10 pliegues y GroupKFold de 5. Como cada pliegue de LOO
tiene una sola canción, las métricas se calculan sobre todas las predicciones fuera de pliegue
(`cross_val_predict`) en los cuatro esquemas, para que sean comparables. El **desempeño real** de un
modelo entrenado con esas 500 canciones se mide en las 14.348 canciones de validación.

In [ ]:
N_LOO = 500
idx_sub = np.random.RandomState(SEMILLA).choice(len(X_aj), N_LOO, replace=False)
X_sub, g_sub = X_aj.iloc[idx_sub], grupos_aj.iloc[idx_sub]
esquemas_sub = {
    'LeaveOneOut': LeaveOneOut(),
    'KFold 5': KFold(5, shuffle=True, random_state=SEMILLA),
    'KFold 10': KFold(10, shuffle=True, random_state=SEMILLA),
    'GroupKFold 5': GroupKFold(5),
}
filas_B = []
for tarea, cfg in TAREAS.items():
    y_sub = cfg['y_aj'].iloc[idx_sub]
    clases = np.unique(y_sub) if tarea == 'clasificacion' else None
    metodo = 'predict' if tarea == 'regresion' else 'predict_proba'
    for nombre, cv in esquemas_sub.items():
        def calcular(cv=cv):
            pred = cross_val_predict(crear_pipeline(clone(cfg['modelo_base'])), X_sub, y_sub, groups=g_sub,
                                     cv=cv, method=metodo, n_jobs=4)
            return metricas_desde_prediccion(tarea, y_sub, pred, clases)
        r = con_cache(f'B_{tarea}_{slug(nombre)}', calcular)
        filas_B.append({'tarea': cfg['nombre'], 'esquema': nombre,
                        'ajustes_del_modelo': cv.get_n_splits(X_sub, y_sub, g_sub), **r})
    real = con_cache(f'B_{tarea}_real_500',
                     lambda: metricas_validacion(tarea, crear_pipeline(clone(cfg['modelo_base'])).fit(X_sub, y_sub),
                                                 X_val, cfg['y_val']))
    filas_B.append({'tarea': cfg['nombre'], 'esquema': 'Real (500 canciones → validación)',
                    'ajustes_del_modelo': 1, **real})

resultados_B = pd.DataFrame(filas_B)
resultados_B.to_csv(PROCESSED_DIR / 'ep3_bloque_B_loo.csv', index=False)
for tarea, cfg in TAREAS.items():
    columnas = ['esquema', 'ajustes_del_modelo'] + list(cfg['scoring']) + ['segundos']
    columnas = [c for c in columnas if c in resultados_B.columns]
    print(f"\n{cfg['nombre']} — submuestra de {N_LOO} canciones")
    display(resultados_B[resultados_B['tarea'] == cfg['nombre']][columnas].set_index('esquema').round(4))

segundos_loo = resultados_B.loc[resultados_B['esquema'] == 'LeaveOneOut', 'segundos'].to_numpy()
print(f'\nCosto estimado de LOO con las {len(X_aj):,} canciones de ajuste: ~{len(X_aj)} ajustes por configuración; '
      f'con ~10 s por ajuste, ≈ {len(X_aj) * 10 / 86400:.1f} días por configuración.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for ax, (tarea, cfg) in zip(axes, TAREAS.items()):
    p = cfg['principal']
    sub = resultados_B[resultados_B['tarea'] == cfg['nombre']]
    estim = sub[sub['esquema'] != 'Real (500 canciones → validación)']
    real = sub.loc[sub['esquema'] == 'Real (500 canciones → validación)', p].iloc[0]
    ax.bar(estim['esquema'], estim[p], color=['tab:purple', 'tab:blue', 'tab:blue', 'tab:green'], alpha=0.8)
    ax.axhline(real, color='black', linestyle='--', label=f'Real en validación = {real:.3f}')
    for x, (v, s, n) in enumerate(zip(estim[p], estim['segundos'], estim['ajustes_del_modelo'])):
        ax.text(x, v, f'{v:.3f}\n{n} ajustes · {s:.0f} s', ha='center', va='bottom', fontsize=8)
    lo, hi = min(estim[p].min(), real), max(estim[p].max(), real)
    ax.set_ylim(lo - (hi - lo) * 1.5 - 0.01, hi + (hi - lo) * 1.5 + 0.01)
    ax.set_title(f"{cfg['nombre']}: {p} estimado con 500 canciones", fontweight='bold')
    ax.set_ylabel(p); ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(IMAGES_DIR / '31_ep3_leave_one_out.png', dpi=150, bbox_inches='tight')
plt.show()

<!-- INTERP_B -->

## 5. Bloque C — GridSearchCV vs. RandomizedSearchCV en cada esquema (Boosting)

Los dos métodos tienen **el mismo presupuesto: 8 configuraciones × 3 pliegues**, y parten de los
hiperparámetros de la EP2.

| Hiperparámetro de HistGradientBoosting | Qué controla | GridSearchCV (2 × 2 × 2 = 8) | RandomizedSearchCV (8 sorteos) |
|---|---|---|---|
| `learning_rate` | Cuánto corrige cada árbol nuevo | {0,03; 0,06} | log-uniforme [0,02; 0,15] |
| `max_leaf_nodes` | Complejidad de cada árbol | {31; 63} | entero [15; 127] |
| `min_samples_leaf` | Mínimo de canciones por hoja (regulariza) | {20; 50} | entero [10; 100] |
| `l2_regularization` | Penaliza hojas con valores extremos | fijo en el valor EP2 | uniforme [0; 1,5] |
| `max_iter` | Máximo de árboles (con parada temprana) | fijo en el valor EP2 | fijo en el valor EP2 |

La grilla prueba combinaciones exactas de pocos valores; la búsqueda aleatoria, con el mismo
costo, explora además `l2_regularization` y valores intermedios.

In [ ]:
ESPACIOS = {
    ('Boosting', 'GridSearchCV'): {
        'modelo__learning_rate': [0.03, 0.06],
        'modelo__max_leaf_nodes': [31, 63],
        'modelo__min_samples_leaf': [20, 50],
    },
    ('Boosting', 'RandomizedSearchCV'): {
        'modelo__learning_rate': loguniform(0.02, 0.15),
        'modelo__max_leaf_nodes': randint(15, 128),
        'modelo__min_samples_leaf': randint(10, 101),
        'modelo__l2_regularization': uniform(0.0, 1.5),
    },
    ('Bagging', 'GridSearchCV'): {
        'modelo__n_estimators': [5, 10],
        'modelo__max_samples': [0.5, 0.8],
    },
    ('Bagging', 'RandomizedSearchCV'): {
        'modelo__n_estimators': randint(5, 13),
        'modelo__max_samples': uniform(0.4, 0.5),
        'modelo__max_features': uniform(0.7, 0.3),
    },
}
N_ITER = {'Boosting': 8, 'Bagging': 4}
DESCRIPCION_ESPACIO = {
    ('Boosting', 'GridSearchCV'): 'learning_rate {0,03; 0,06} × max_leaf_nodes {31; 63} × min_samples_leaf {20; 50}',
    ('Boosting', 'RandomizedSearchCV'): '8 sorteos: learning_rate log-U[0,02; 0,15], max_leaf_nodes [15; 127], '
                                        'min_samples_leaf [10; 100], l2_regularization U[0; 1,5]',
    ('Bagging', 'GridSearchCV'): 'n_estimators {5; 10} × max_samples {0,5; 0,8}',
    ('Bagging', 'RandomizedSearchCV'): '4 sorteos: n_estimators [5; 12], max_samples U[0,4; 0,9], '
                                       'max_features U[0,7; 1,0]',
}
MEMORIA = joblib.Memory(tempfile.mkdtemp(prefix='ep3_prep_'), verbose=0)


def crear_modelo(tarea, estrategia):
    cfg = TAREAS[tarea]
    if estrategia == 'Boosting':
        return clone(cfg['modelo_base'])
    return cfg['bagging'](estimator=clone(cfg['modelo_base']), bootstrap=True, n_jobs=1, random_state=SEMILLA)


def ejecutar_busqueda(tarea, estrategia, metodo, splits):
    cfg = TAREAS[tarea]
    pipe = crear_pipeline(crear_modelo(tarea, estrategia), MEMORIA)
    espacio = ESPACIOS[(estrategia, metodo)]
    comunes = dict(scoring=cfg['scoring'], refit=cfg['principal'], cv=splits, n_jobs=1, error_score='raise')
    if metodo == 'GridSearchCV':
        busqueda = GridSearchCV(pipe, espacio, **comunes)
    else:
        busqueda = RandomizedSearchCV(pipe, espacio, n_iter=N_ITER[estrategia], random_state=SEMILLA, **comunes)
    busqueda.fit(X_aj, cfg['y_aj'])
    MEMORIA.clear(warn=False)
    res = pd.DataFrame(busqueda.cv_results_)
    detalle = pd.DataFrame({'hiperparametros': res['params'].map(formatear_params),
                            **{f'{m}_cv': signo(cfg['scoring'], m) * res[f'mean_test_{m}'] for m in cfg['scoring']},
                            'segundos_ajuste': res['mean_fit_time']})
    mejor = res.iloc[busqueda.best_index_]
    return {'mejores_parametros': {k.replace('modelo__', ''): v for k, v in busqueda.best_params_.items()},
            **{f'{m}_cv': signo(cfg['scoring'], m) * mejor[f'mean_test_{m}'] for m in cfg['scoring']},
            **{f'{m}_std': mejor[f'std_test_{m}'] for m in cfg['scoring']},
            'configuraciones': len(res), 'detalle': detalle,
            'validacion': metricas_validacion(tarea, busqueda.best_estimator_, X_val, cfg['y_val'])}


def fila_busqueda(tarea, estrategia, metodo, esquema, r, fuga):
    cfg = TAREAS[tarea]
    return {'tarea': cfg['nombre'], 'estrategia': estrategia, 'metodo': metodo, 'esquema': esquema,
            'fuga_%': fuga, 'configuraciones': r['configuraciones'],
            'espacio': DESCRIPCION_ESPACIO[(estrategia, metodo)],
            'hiperparametros_elegidos': formatear_params(r['mejores_parametros']),
            'params': r['mejores_parametros'],
            **{f'{m}_cv': r[f'{m}_cv'] for m in cfg['scoring']},
            f"{cfg['principal']}_std": r[f"{cfg['principal']}_std"],
            **{f'val_{m}': v for m, v in r['validacion'].items()}, 'segundos': r['segundos']}


ESQUEMAS_BUSQUEDA = ['KFold (estándar)', 'StratifiedKFold (estratificado)', 'GroupKFold (agrupado)']
busquedas = {}
filas_C = []
for tarea in TAREAS:
    esquemas = crear_esquemas(tarea)
    for esquema in ESQUEMAS_BUSQUEDA:
        for metodo in ['GridSearchCV', 'RandomizedSearchCV']:
            clave = f'C_{tarea}_boosting_{slug(metodo)}_{slug(esquema)}'
            r = con_cache(clave, lambda: ejecutar_busqueda(tarea, 'Boosting', metodo, esquemas[esquema]))
            busquedas[clave] = r
            filas_C.append(fila_busqueda(tarea, 'Boosting', metodo, esquema, r, porcentaje_fuga(esquemas[esquema])))
resultados_C = pd.DataFrame(filas_C)
print('Bloque C completo.')

In [ ]:
for tarea, cfg in TAREAS.items():
    p, s = cfg['principal'], cfg['secundaria']
    sub = resultados_C[resultados_C['tarea'] == cfg['nombre']]
    print(f"\n{cfg['nombre']} — Boosting: hiperparámetros elegidos por cada método y esquema")
    display(sub[['metodo', 'esquema', 'fuga_%', 'hiperparametros_elegidos', f'{p}_cv', f'val_{p}', f'val_{s}',
                 'segundos']].set_index(['metodo', 'esquema']).round(4))

# Ejemplo de todas las configuraciones probadas por una búsqueda (regresión, GroupKFold)
for metodo in ['GridSearchCV', 'RandomizedSearchCV']:
    print(f'\nRegresión — {metodo} con GroupKFold: las 8 configuraciones evaluadas')
    display(busquedas[f'C_regresion_boosting_{slug(metodo)}_groupkfold_agrupado']['detalle']
            .sort_values('MAE_cv').round(4).reset_index(drop=True))

<!-- INTERP_C -->

## 6. Bloque D — Bagging vs. Boosting

Se envuelve el HistGradientBoosting de la EP2 (con sus hiperparámetros) en un `Bagging`: se
entrenan varias copias, cada una con una muestra bootstrap distinta (y, en la búsqueda aleatoria,
con una fracción de las variables), y se promedian sus predicciones o probabilidades. Se usa
GroupKFold, el esquema sin fuga, con ambos métodos de búsqueda.

| Hiperparámetro de Bagging | Qué controla | GridSearchCV (2 × 2 = 4) | RandomizedSearchCV (4 sorteos) |
|---|---|---|---|
| `n_estimators` | Cantidad de modelos promediados | {5; 10} | entero [5; 12] |
| `max_samples` | Fracción de canciones de cada muestra bootstrap | {0,5; 0,8} | uniforme [0,4; 0,9] |
| `max_features` | Fracción de variables que ve cada modelo | 1,0 (todas) | uniforme [0,7; 1,0] |

Cada configuración de Bagging cuesta entre 5 y 12 entrenamientos del modelo base; por eso el
presupuesto es de 4 configuraciones por método.

In [ ]:
filas_D = []
for tarea in TAREAS:
    splits = crear_esquemas(tarea)['GroupKFold (agrupado)']
    for metodo in ['GridSearchCV', 'RandomizedSearchCV']:
        clave = f'D_{tarea}_bagging_{slug(metodo)}_groupkfold'
        r = con_cache(clave, lambda: ejecutar_busqueda(tarea, 'Bagging', metodo, splits))
        busquedas[clave] = r
        filas_D.append(fila_busqueda(tarea, 'Bagging', metodo, 'GroupKFold (agrupado)', r, porcentaje_fuga(splits)))
resultados_D = pd.DataFrame(filas_D)

for tarea, cfg in TAREAS.items():
    p, s = cfg['principal'], cfg['secundaria']
    sub = resultados_D[resultados_D['tarea'] == cfg['nombre']]
    print(f"\n{cfg['nombre']} — Bagging sobre HistGradientBoosting")
    display(sub[['metodo', 'hiperparametros_elegidos', f'{p}_cv', f'val_{p}', f'val_{s}', 'segundos']]
            .set_index('metodo').round(4))
    for metodo in ['GridSearchCV', 'RandomizedSearchCV']:
        print(f'  Configuraciones de {metodo}:')
        display(busquedas[f'D_{tarea}_bagging_{slug(metodo)}_groupkfold']['detalle'].round(4))

<!-- INTERP_D -->

## 7. Comparación de todas las combinaciones y elección

Se reúnen, por tarea, las 9 combinaciones (referencia EP2 con `cross_validate`, 6 de Boosting y 2
de Bagging). La referencia usa la estimación de GroupKFold del bloque A. El orden lo da la métrica
principal en **validación**, la vara común (ver 1.2).

In [ ]:
def fila_referencia(tarea):
    cfg = TAREAS[tarea]
    a = resultados_A[(resultados_A['tarea'] == cfg['nombre']) & (resultados_A['esquema'] == 'GroupKFold (agrupado)')].iloc[0]
    return {'tarea': cfg['nombre'], 'estrategia': 'Boosting', 'metodo': 'CrossValidation (sin búsqueda)',
            'esquema': 'GroupKFold (agrupado)', 'fuga_%': a['fuga_%'], 'configuraciones': 1,
            'espacio': 'Hiperparámetros fijos de la EP2', 'hiperparametros_elegidos': formatear_params(cfg['params_ep2']),
            'params': {}, **{f'{m}_cv': a[f'{m}_cv'] for m in cfg['scoring']},
            f"{cfg['principal']}_std": a[f"{cfg['principal']}_std"],
            **{f'val_{m}': v for m, v in referencia_val[tarea].items() if m != 'segundos'}, 'segundos': a['segundos']}


tablas_finales, ganadores = {}, {}
for tarea, cfg in TAREAS.items():
    p, s = cfg['principal'], cfg['secundaria']
    tabla = pd.DataFrame([fila_referencia(tarea)]
                         + resultados_C[resultados_C['tarea'] == cfg['nombre']].to_dict('records')
                         + resultados_D[resultados_D['tarea'] == cfg['nombre']].to_dict('records'))
    decimales = 2 if tarea == 'regresion' else 3
    tabla['_principal'] = tabla[f'val_{p}'].round(decimales) * (-1 if cfg['mayor_es_mejor'] else 1)
    tabla['_secundaria'] = tabla[f'val_{s}'] * (-1 if cfg['mayor_es_mejor'] else 1)
    tabla = tabla.sort_values(['_principal', '_secundaria', 'segundos']).reset_index(drop=True)
    tabla.index = tabla.index + 1
    tabla.index.name = 'ranking'
    ref = tabla.loc[tabla['metodo'] == 'CrossValidation (sin búsqueda)', f'val_{p}'].iloc[0]
    tabla[f'mejora_vs_EP2_{p}'] = (tabla[f'val_{p}'] - ref) * (1 if cfg['mayor_es_mejor'] else -1)
    tablas_finales[tarea] = tabla
    ganadores[tarea] = tabla.iloc[0]
    columnas = ['estrategia', 'metodo', 'esquema', 'configuraciones', 'hiperparametros_elegidos',
                f'{p}_cv', f'val_{p}', f'val_{s}', f'mejora_vs_EP2_{p}', 'segundos']
    print(f"\n{cfg['nombre']} — ranking por {p} en validación")
    display(tabla[columnas].round(4))
    g = ganadores[tarea]
    print(f"GANADORA: {g['estrategia']} + {g['metodo']} + {g['esquema']} -> {g['hiperparametros_elegidos']}")
    print(f"  {p} validación = {g[f'val_{p}']:.4f} (EP2: {ref:.4f}; mejora = {g[f'mejora_vs_EP2_{p}']:+.4f})")

pd.concat(tablas_finales.values()).drop(columns=['params', '_principal', '_secundaria']).to_csv(
    PROCESSED_DIR / 'ep3_comparacion_combinaciones.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, (tarea, cfg) in zip(axes, TAREAS.items()):
    p = cfg['principal']
    t = tablas_finales[tarea].iloc[::-1]
    etiquetas = [f"{e} · {m.replace('SearchCV', '').replace('CrossValidation (sin búsqueda)', 'CV EP2')}"
                 f" · {q.split(' (')[0]}" for e, m, q in zip(t['estrategia'], t['metodo'], t['esquema'])]
    y = np.arange(len(t))
    ax.barh(y - 0.2, t[f'{p}_cv'], height=0.4, color='lightgray', label=f'{p} estimado por CV')
    ax.barh(y + 0.2, t[f'val_{p}'], height=0.4, color='tab:blue', label=f'{p} en validación (vara común)')
    ax.barh(len(t) - 1 + 0.2, t[f'val_{p}'].iloc[-1], height=0.4, color='tab:green', label='Ganadora')
    ax.set_yticks(y); ax.set_yticklabels(etiquetas, fontsize=8)
    valores = pd.concat([t[f'{p}_cv'], t[f'val_{p}']])
    rango = valores.max() - valores.min()
    ax.set_xlim(valores.min() - rango * 0.3, valores.max() + rango * 0.3)
    ax.set_title(f"{cfg['nombre']}: {p} por combinación (arriba = mejor)", fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig(IMAGES_DIR / '32_ep3_comparacion_combinaciones.png', dpi=150, bbox_inches='tight')
plt.show()

<!-- INTERP_SEL -->

## 8. Evaluación final en prueba

La combinación ganadora de cada tarea se reentrena con las **71.740 canciones** de entrenamiento y se
evalúa **una sola vez** en el conjunto de prueba de la EP2. Se compara con los modelos de la EP2
guardados en `models/`, medidos con el mismo código.

En clasificación, el umbral de la clase "alto" se recalcula para el modelo ganador con la misma regla
de la EP2: el mayor umbral *u* que logra recall de "alto" ≥ 0,60 con probabilidades fuera de pliegue
del entrenamiento (GroupKFold de 3 pliegues). El conjunto de prueba no participa.

In [ ]:
def modelo_ganador(tarea):
    g = ganadores[tarea]
    modelo = crear_modelo(tarea, g['estrategia'])
    if g['params']:
        modelo.set_params(**g['params'])
    return crear_pipeline(modelo)


def predecir_con_umbral(proba, umbral, clases):
    clases = list(clases)
    i_alto = clases.index('alto')
    resto = [i for i in range(len(clases)) if i != i_alto]
    pred = np.array(clases, dtype=object)[np.array(resto)[proba[:, resto].argmax(axis=1)]]
    return np.where(proba[:, i_alto] >= umbral, 'alto', pred)


filas_test = []
modelos_ep3 = {}
for tarea, cfg in TAREAS.items():
    es_referencia = ganadores[tarea]['metodo'] == 'CrossValidation (sin búsqueda)'

    def entrenar():
        modelo = modelo_ganador(tarea).fit(X_train, cfg['y_train'])
        salida = {'modelo': modelo}
        if tarea == 'clasificacion':
            proba_oof = cross_val_predict(modelo_ganador(tarea), X_train, cfg['y_train'], groups=grupos_train,
                                          cv=GroupKFold(N_PLIEGUES), method='predict_proba')
            umbrales = np.round(np.arange(0.05, 0.71, 0.01), 2)
            recalls = [recall_score(cfg['y_train'], predecir_con_umbral(proba_oof, u, modelo.classes_),
                                    labels=['alto'], average='macro') for u in umbrales]
            validos = [u for u, r in zip(umbrales, recalls) if r >= 0.60]
            salida['umbral'] = float(max(validos)) if validos else float(umbrales[int(np.argmax(recalls))])
        return salida

    r = con_cache(f'F_{tarea}_ganador_final', entrenar)
    modelos_ep3[tarea] = r
    for nombre, modelo in [('EP2 (models/)', modelo_ep2[tarea]), ('EP3 ganador', r['modelo'])]:
        fila = {'tarea': cfg['nombre'], 'modelo': nombre, **metricas_validacion(tarea, modelo, X_test, cfg['y_test'])}
        if tarea == 'clasificacion':
            umbral = (joblib.load(MODELS_DIR / 'umbral_clase_alto.joblib')['umbral_alto']
                      if nombre.startswith('EP2') else r['umbral'])
            pred_u = predecir_con_umbral(modelo.predict_proba(X_test), umbral, modelo.classes_)
            fila.update({'umbral_alto': umbral,
                         'Recall_alto_umbral': recall_score(cfg['y_test'], pred_u, labels=['alto'], average='macro'),
                         'Precision_alto_umbral': precision_score(cfg['y_test'], pred_u, labels=['alto'], average='macro'),
                         'F1_macro_umbral': f1_score(cfg['y_test'], pred_u, average='macro')})
        filas_test.append(fila)

resultados_test = pd.DataFrame(filas_test)
resultados_test.to_csv(PROCESSED_DIR / 'ep3_resultados_prueba.csv', index=False)
for tarea, cfg in TAREAS.items():
    sub = resultados_test[resultados_test['tarea'] == cfg['nombre']].dropna(axis=1, how='all')
    print(f"\n{cfg['nombre']} — conjunto de prueba (17.843 canciones)")
    display(sub.drop(columns='tarea').set_index('modelo').round(4))
    g = ganadores[tarea]
    print(f"Ganadora EP3: {g['estrategia']} + {g['metodo']} + {g['esquema']} -> {g['hiperparametros_elegidos']}")

In [ ]:
# Se guardan los modelos ganadores solo si difieren de la configuración de la EP2
for tarea, cfg in TAREAS.items():
    if ganadores[tarea]['metodo'] == 'CrossValidation (sin búsqueda)':
        print(f"{cfg['nombre']}: la ganadora es la configuración de la EP2; se mantiene models/ sin cambios.")
        continue
    nombre = {'regresion': 'regresor_popularidad_ep3.joblib', 'clasificacion': 'clasificador_popularidad_ep3.joblib'}[tarea]
    joblib.dump(modelos_ep3[tarea]['modelo'], MODELS_DIR / nombre, compress=('xz', 3))
    print(f"{cfg['nombre']}: guardado models/{nombre} ({(MODELS_DIR / nombre).stat().st_size / 1e6:.1f} MB)")
    if tarea == 'clasificacion':
        joblib.dump({'umbral_alto': modelos_ep3[tarea]['umbral'],
                     'regla': "'alto' si P(alto) >= umbral; si no, la más probable entre 'bajo' y 'medio'"},
                    MODELS_DIR / 'umbral_clase_alto_ep3.joblib')
        print('  y models/umbral_clase_alto_ep3.joblib')

<!-- INTERP_TEST -->

<!-- CONCLUSIONES -->